# Scenario 10: Pure OpenTelemetry Integration with Galileo

## What You'll Learn

This notebook shows the **pure OpenTelemetry** approach to sending traces to Galileo — zero vendor-specific code in the tracing path. Galileo is treated as just another OTLP-compatible backend, like Jaeger, Datadog, or Honeycomb.

By the end, you'll understand how to:

1. **Configure OTel via environment variables** (the OTel-standard way)
2. **Set up a `TracerProvider`** with `BatchSpanProcessor` + `OTLPSpanExporter`
3. **Attach a `Resource`** with `service.name` and `deployment.environment` (OTel best practice)
4. **Auto-instrument OpenAI** with the OTel contrib `opentelemetry-instrumentation-openai-v2` package
5. **Use OTel's own GenAI semantic conventions** (`gen_ai.*` attributes) — no third-party semconv
6. **Combine all span types in one trace** — an Agent+RAG workflow with nested Retriever, LLM, and Tool spans
7. **Build reusable decorators** for clean pipelines using only standard OTel APIs

## Why Pure OTel?

Previous versions of this notebook used Galileo's `GalileoSpanProcessor` and `start_galileo_span` (vendor-specific) or OpenInference's `OpenAIInstrumentor` and `SpanAttributes` (third-party semconv). Both work, but they couple your code to an ecosystem.

The **pure-OTel** approach has one rule: **every import in the tracing path comes from `opentelemetry.*`**. Vendor details (endpoint, auth) are configuration, not code.

```
┌──────────────────────────────────────────┐
│ Application Code                          │
│ (uses only opentelemetry.* APIs)         │
└──────────────┬───────────────────────────┘
               │
        TracerProvider + Resource
               │
        BatchSpanProcessor
               │
        OTLPSpanExporter
               │
               ▼
   OTEL_EXPORTER_OTLP_TRACES_ENDPOINT  ──►  Galileo
   OTEL_EXPORTER_OTLP_TRACES_HEADERS         (or any OTLP backend)
```

## Best Practices for OTel + Single-Vendor Backend

| Practice | Why |
|----------|-----|
| **Configure via env vars** (`OTEL_EXPORTER_OTLP_*`, `OTEL_SERVICE_NAME`) | Deploy-time config without code changes |
| **Use standard OTLP exporter** | Portable across backends |
| **Follow OTel semantic conventions** (`gen_ai.*`) | Vendors compete on ingestion, not lock-in |
| **Attach a `Resource`** (`service.name`, `service.version`, `deployment.environment`) | Correlate traces across services |
| **Set global tracer provider** | Libraries and instrumentors auto-discover it |
| **`BatchSpanProcessor` in prod, `SimpleSpanProcessor` for debugging** | Throughput vs. immediacy |
| **Explicit `force_flush()` + `shutdown()`** | No lost spans on exit |

## Key Concepts

| Concept | What it means |
|---------|---------------|
| **TracerProvider** | The OTel tracing engine — creates tracers and owns span processors |
| **Resource** | Metadata describing the emitting service (name, version, environment) |
| **BatchSpanProcessor** | Buffers spans and exports in batches (production) |
| **OTLPSpanExporter** | Standard exporter — sends spans over OTLP/HTTP to any compatible endpoint |
| **LoggerProvider** | Owns the logs pipeline — the OTel OpenAI instrumentor emits message content as log events here |
| **SpanAttrBridgeProcessor** | Custom `LogRecordProcessor` that copies message content onto the active span as `gen_ai.input.messages` / `gen_ai.output.messages` |
| **GenAI semconv** | OTel's standard `gen_ai.*` attributes for LLM telemetry |

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY`
- Dependencies installed via `uv sync`

## Step 0: Load Environment Variables

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


## Step 0b: Imports and Constants

Every tracing import comes from `opentelemetry.*`. No OpenInference. No Galileo in the tracing path.

| Import | Purpose |
|--------|---------|
| `opentelemetry.sdk.trace.TracerProvider` | Tracing engine |
| `opentelemetry.sdk.trace.export.BatchSpanProcessor` | Batched span export |
| `opentelemetry.exporter.otlp.proto.http.trace_exporter.OTLPSpanExporter` | OTLP/HTTP exporter |
| `opentelemetry.sdk.resources.Resource` | Service-level metadata |
| `opentelemetry.instrumentation.openai_v2.OpenAIInstrumentor` | OTel-contrib OpenAI auto-instrumentation |
| `opentelemetry.sdk._logs.LoggerProvider` / `LogRecordProcessor` | Logs pipeline — catches message-content events from the instrumentor |
| `opentelemetry._logs.set_logger_provider` | Registers the logger provider globally |
| `opentelemetry.semconv._incubating.attributes.gen_ai_attributes` | OTel GenAI semantic conventions (`gen_ai.*`) |

The only Galileo imports are for bootstrapping (`galileo_context.init()` for project/log-stream creation and `GalileoPythonConfig` for config lookup). None touch the tracing path.

In [2]:
import os

import openai
from galileo import galileo_context
from galileo.config import GalileoPythonConfig
from galileo.projects import delete_project
from opentelemetry import trace
from opentelemetry._logs import set_logger_provider
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.semconv._incubating.attributes import gen_ai_attributes as gen_ai

PROJECT_NAME = 'GalileoEval_S10_OTel'
LOG_STREAM_NAME = 'otel-integration'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, MODEL

('GalileoEval_S10_OTel', 'otel-integration', 'gpt-4o-mini')

## Step 1: Initialize Galileo

Same as every other notebook — `galileo_context.init()` authenticates, creates the project and log stream, and initializes the config. The OTel-specific setup comes next.

In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8
https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8/log-streams/7e8bce39-8429-4f4d-be58-42a7e894e931


## Step 2: OTel Pipeline — Traces + Content-to-Span Bridge

This is the OTel-standard pattern for GenAI, with one caveat we need to handle.

### Two signals in the GenAI spec

1. **Traces pipeline** (`TracerProvider` → `OTLPSpanExporter`) — span metadata: span name, timings, `gen_ai.*` attributes (model, tokens, operation name).
2. **Logs pipeline** (`LoggerProvider` → log exporter) — message content emitted as structured log events (`gen_ai.user.message`, `gen_ai.choice`, ...).

### The caveat with Galileo

Galileo's OTLP endpoint accepts **trace** payloads but rejects OTLP **log** payloads (returns 422 because the protobuf schema is different). Galileo's Console reads input/output from span **attributes** — the older convention — not from log events.

### The bridge

We keep a `LoggerProvider` (required to enable content capture at all), but instead of exporting log records to Galileo, we use a **custom `LogRecordProcessor`** that intercepts each log event and copies the content onto the **active span as attributes**:

```
OpenAIInstrumentor
    │
    ├─► span  (gen_ai.* metadata)  ──► OTLPSpanExporter  ──► Galileo
    │
    └─► log events (content)       ──► SpanAttrBridge    ──► same span's attributes
                                         (gen_ai.input.messages,
                                          gen_ai.output.messages)
```

This keeps everything pure OTel on the instrumentation side, while giving Galileo what it needs to render input/output.

The bridge accumulates messages per span (keyed by `span_id`) so multi-turn prompts build up correctly, serializing to JSON on every event:

```python
[{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
```

Note that the same `Resource` is attached to both the `TracerProvider` and `LoggerProvider` — every span and event carries the same service identity.

### Env vars

```bash
OTEL_EXPORTER_OTLP_TRACES_ENDPOINT=https://api.galileo.ai/otel/traces
OTEL_EXPORTER_OTLP_TRACES_HEADERS=Galileo-API-Key=<key>,project=<name>,logstream=<name>
OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=true   # ← opt-in content capture
OTEL_SERVICE_NAME=my-service
```

> **Gotcha:** use the TRACES-specific env vars (`OTEL_EXPORTER_OTLP_TRACES_*`), not the generic `OTEL_EXPORTER_OTLP_ENDPOINT` — the generic one auto-appends `/v1/traces`, which 404s on Galileo.

In [4]:
import json
from collections import defaultdict

from opentelemetry.sdk._logs import LogRecordProcessor


class SpanAttrBridgeProcessor(LogRecordProcessor):
    """Bridge OTel GenAI log events into Galileo-compatible span attributes.

    The opentelemetry-instrumentation-openai-v2 package emits message content as
    log events (gen_ai.user.message, gen_ai.assistant.message, gen_ai.choice, ...).
    Galileo's Console reads prompts/completions from span *attributes*:

      - gen_ai.input.messages   →  JSON-encoded list of {role, content} dicts
      - gen_ai.output.messages  →  JSON-encoded list of {role, content} dicts

    This processor accumulates the emitted log events (keyed by span_id) and
    rewrites the two attributes as each new event arrives.
    """

    def __init__(self):
        # span_id -> {"input": [...], "output": [...]}
        self._buffers = defaultdict(lambda: {'input': [], 'output': []})

    def on_emit(self, record):
        try:
            span = trace.get_current_span()
            if not span or not span.is_recording():
                return
            span_id = span.get_span_context().span_id
            event_name = record.log_record.event_name or ''
            body = record.log_record.body
            if not isinstance(body, dict):
                return
            buf = self._buffers[span_id]

            # Input-side events: gen_ai.{system,user,assistant,tool}.message
            if event_name.startswith('gen_ai.') and event_name.endswith('.message'):
                role = event_name.split('.')[1]  # system | user | assistant | tool
                content = body.get('content')
                if content is not None:
                    buf['input'].append({'role': role, 'content': str(content)})
                    span.set_attribute('gen_ai.input.messages', json.dumps(buf['input']))

            # Output-side event: gen_ai.choice
            elif event_name == 'gen_ai.choice':
                msg = body.get('message') or {}
                if isinstance(msg, dict) and msg.get('content'):
                    buf['output'].append({
                        'role': msg.get('role', 'assistant'),
                        'content': str(msg['content']),
                    })
                    span.set_attribute('gen_ai.output.messages', json.dumps(buf['output']))
        except Exception:
            # Processor must never raise — OTel spec requirement.
            pass

    def shutdown(self):
        pass

    def force_flush(self, timeout_millis=30000):
        return True


# 1. Derive Galileo endpoint + auth from the already-initialized config
config = GalileoPythonConfig.get()
api_url = str(config.api_url)
if not api_url.endswith('/'):
    api_url += '/'
otlp_endpoint = f'{api_url}otel/traces'
otlp_headers = {
    'Galileo-API-Key': config.api_key.get_secret_value(),
    'project': PROJECT_NAME,
    'logstream': LOG_STREAM_NAME,
}

# 2. Standard OTel env vars (use TRACES-specific form to avoid auto /v1/traces append)
os.environ['OTEL_EXPORTER_OTLP_TRACES_ENDPOINT'] = otlp_endpoint
os.environ['OTEL_EXPORTER_OTLP_TRACES_HEADERS'] = ','.join(f'{k}={v}' for k, v in otlp_headers.items())
os.environ['OTEL_SERVICE_NAME'] = 'galileo-otel-demo'

# 3. Opt in to content capture (default is false for privacy)
os.environ['OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT'] = 'true'

# 4. Resource — describes the emitting service
resource = Resource.create({
    'service.name': 'galileo-otel-demo',
    'service.version': '1.0.0',
    'deployment.environment': 'notebook',
})

# 5. Traces pipeline: TracerProvider → BatchSpanProcessor → OTLPSpanExporter
tracer_provider = TracerProvider(resource=resource)
tracer_provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(tracer_provider)

# 6. Logs pipeline: LoggerProvider with the SpanAttrBridgeProcessor (no network export)
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(SpanAttrBridgeProcessor())
set_logger_provider(logger_provider)

# 7. Auto-instrument OpenAI. Pass BOTH providers — the logger is how content reaches us.
OpenAIInstrumentor().instrument(
    tracer_provider=tracer_provider,
    logger_provider=logger_provider,
)

# 8. Standard OpenAI client and tracer
client = openai.OpenAI()
tracer = trace.get_tracer(__name__)

print('Pure OTel pipeline ready:')
print(f'  Traces  → {otlp_endpoint}')
print(f'  Content → bridged to gen_ai.input.messages / gen_ai.output.messages on span')
print(f'  Content capture: {os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"]}')

Pure OTel pipeline ready:
  Traces  → https://api.demo-v2.galileocloud.io/otel/traces
  Content → bridged to gen_ai.input.messages / gen_ai.output.messages on span
  Content capture: true


## Step 3: Make an LLM Call

A simple chat completion. Because OpenAI is auto-instrumented, this call automatically creates an OTel span with `gen_ai.*` metadata (model, tokens, finish-reason). Message content is delivered separately as log events, which our `SpanAttrBridgeProcessor` copies onto the span as `gen_ai.input.messages` / `gen_ai.output.messages`.

### Why we flush BOTH pipelines

`BatchSpanProcessor` buffers spans and exports them on a background thread — by default every **5 seconds** or when the batch fills (512 spans). That's ideal for production throughput, but annoying in a notebook when you want to see the trace in Galileo immediately.

`tracer_provider.force_flush()` forces immediate span export. `logger_provider.force_flush()` drains any pending log events through the bridge. We flush both after each step so traces + content appear in the Console as soon as a cell finishes. In production you'd only flush on shutdown.

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant that explains technical concepts clearly.'},
        {'role': 'user', 'content': 'What is OpenTelemetry and why is it useful for AI applications?'},
    ],
    max_tokens=200,
)

# Flush BOTH pipelines — span metadata AND the message-content log events.
tracer_provider.force_flush()
logger_provider.force_flush()

print('Response:')
print(response.choices[0].message.content)
print(f'\nTokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out')
print(f'\n\u2192 Trace + content log events flushed to Galileo. Check: {log_stream_url}')

Response:
OpenTelemetry is an open-source observability framework designed to help developers collect, process, and export telemetry data (like metrics, logs, and traces) from their applications. It provides a standardized way to instrument code, gather data from various services, and send that data to different back-end observability platforms for analysis. OpenTelemetry is part of the Cloud Native Computing Foundation (CNCF) and aims to support various programming languages and environments.

### Key Features of OpenTelemetry:

1. **Unified Approach**: OpenTelemetry combines monitoring, logging, and tracing into a single framework, allowing developers to have an integrated view of their applications' performance.
  
2. **Automatic Instrumentation**: It provides libraries and toolkits to automatically instrument applications, reducing the need for manual code changes.

3. **Compatibility with Multiple Backends**: Data collected can be sent to multiple observability backends like Prome

## Step 4: Agent + RAG Workflow — All Span Types in One Trace

Real AI apps combine multiple span types in a single request. This step showcases the full OTel GenAI span vocabulary in **one nested trace**:

```
invoke_agent (parent)           ← Agent span: orchestrates the request
  └─ retrieval                  ← Retriever span: vector search for context
  └─ chat (auto-instrumented)   ← LLM span: synthesis using retrieved context
  └─ execute_tool               ← Tool span: post-processing
```

### Which OTel GenAI attributes each span uses

| Span | `gen_ai.operation.name` | Key attributes |
|------|------------------------|----------------|
| Agent | `invoke_agent` | `gen_ai.agent.name`, `gen_ai.agent.id`, `gen_ai.agent.description`, `gen_ai.agent.version` |
| Retriever | `retrieval` | `db.operation = "search"`, structured `gen_ai.output.messages` with `{"documents": [...]}` |
| LLM | `chat` (auto) | Set by `OpenAIInstrumentor` — model, tokens, plus bridged `gen_ai.input/output.messages` |
| Tool | `execute_tool` | `gen_ai.tool.name`, `gen_ai.tool.call.id` |

All four are children of a single request span. The auto-instrumented LLM span sits inside whichever span is active when `client.chat.completions.create` runs — here, the agent span.

In [6]:
user_question = 'What are the best practices for OpenTelemetry in LLM apps?'

# ============================================================
# Agent span (parent) — represents the orchestrator
# ============================================================
with tracer.start_as_current_span('research-agent') as agent_span:
    agent_span.set_attribute(gen_ai.GEN_AI_OPERATION_NAME, gen_ai.GenAiOperationNameValues.INVOKE_AGENT.value)
    agent_span.set_attribute(gen_ai.GEN_AI_AGENT_NAME, 'research-assistant')
    agent_span.set_attribute(gen_ai.GEN_AI_AGENT_ID, 'agent-run-0001')
    agent_span.set_attribute(gen_ai.GEN_AI_AGENT_DESCRIPTION, 'RAG agent that retrieves docs and synthesizes answers.')
    agent_span.set_attribute(gen_ai.GEN_AI_AGENT_VERSION, '1.0.0')
    agent_span.set_attribute(
        'gen_ai.input.messages',
        json.dumps([{'role': 'user', 'content': user_question}]),
    )

    # --------------------------------------------------------
    # Retriever span (child) — vector search
    # --------------------------------------------------------
    with tracer.start_as_current_span('vector-search') as retriever_span:
        retriever_span.set_attribute(gen_ai.GEN_AI_OPERATION_NAME, gen_ai.GenAiOperationNameValues.RETRIEVAL.value)
        retriever_span.set_attribute('db.operation', 'search')
        retriever_span.set_attribute(
            'gen_ai.input.messages',
            json.dumps([{'role': 'user', 'content': user_question}]),
        )
        retrieved = [
            {'content': 'Always use a BatchSpanProcessor in production.', 'metadata': {'source': 'otel-docs', 'score': 0.91}},
            {'content': 'Set OTEL_SERVICE_NAME so spans carry a service identity.', 'metadata': {'source': 'otel-docs', 'score': 0.87}},
            {'content': 'Use TRACES-specific env vars to avoid /v1/traces auto-append.', 'metadata': {'source': 'gotchas', 'score': 0.82}},
        ]
        retriever_span.set_attribute(
            'gen_ai.output.messages',
            json.dumps([{'role': 'assistant', 'content': {'documents': retrieved}}]),
        )
        print(f'Retrieved {len(retrieved)} documents')

    # --------------------------------------------------------
    # LLM span (child, auto-instrumented by OpenAIInstrumentor)
    # The SpanAttrBridgeProcessor attaches message content to this span.
    # --------------------------------------------------------
    context_text = '\n'.join(f'- {d["content"]}' for d in retrieved)
    synthesis = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer concisely based on the provided context.'},
            {'role': 'user', 'content': f'Context:\n{context_text}\n\nQuestion: {user_question}'},
        ],
        max_tokens=150,
    )
    answer = synthesis.choices[0].message.content

    # --------------------------------------------------------
    # Tool span (child) — post-processing
    # --------------------------------------------------------
    with tracer.start_as_current_span('format-final-answer') as tool_span:
        tool_span.set_attribute(gen_ai.GEN_AI_OPERATION_NAME, gen_ai.GenAiOperationNameValues.EXECUTE_TOOL.value)
        tool_span.set_attribute(gen_ai.GEN_AI_TOOL_NAME, 'format-final-answer')
        tool_span.set_attribute(gen_ai.GEN_AI_TOOL_CALL_ID, 'call_format_001')
        tool_span.set_attribute('tool.input', answer)
        formatted = answer.strip()
        tool_span.set_attribute('tool.output', formatted)

    # Record the agent's final output on the parent span
    agent_span.set_attribute(
        'gen_ai.output.messages',
        json.dumps([{'role': 'assistant', 'content': formatted}]),
    )

tracer_provider.force_flush()
logger_provider.force_flush()

print(f'\nFinal answer: {formatted}')
print(f'\n\u2192 One trace exported with 4 span types: agent \u2192 retriever \u2192 llm \u2192 tool')
print(f'  Check: {log_stream_url}')

Retrieved 3 documents

Final answer: The best practices for OpenTelemetry in LLM apps include:

1. Use a BatchSpanProcessor in production to optimize span processing.
2. Set the OTEL_SERVICE_NAME environment variable to ensure that spans carry an appropriate service identity.
3. Utilize TRACES-specific environment variables to prevent automatic appending of /v1/traces to your telemetry data.

→ One trace exported with 4 span types: agent → retriever → llm → tool
  Check: https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8/log-streams/7e8bce39-8429-4f4d-be58-42a7e894e931


## Step 5: Reusable Decorators (Pure OTel)

Every production OTel codebase builds lightweight decorators so span creation doesn't clutter business logic. Here are three that use only `opentelemetry.*` APIs:

- `@otel_tool(name)` — tags a function as an `execute_tool` operation with `gen_ai.tool.name`
- `@otel_retriever(name)` — tags as a retrieval step with custom `retrieval.*` attributes
- `@otel_span(name)` — generic span wrapper for workflows / chains

These are vendor-neutral. Paste them into any OTel codebase and they work.

In [7]:
import functools


def otel_tool(name):
    """Wrap a function as an OTel span tagged with gen_ai.* tool attributes."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            with tracer.start_as_current_span(name) as span:
                span.set_attribute(gen_ai.GEN_AI_OPERATION_NAME, 'execute_tool')
                span.set_attribute(gen_ai.GEN_AI_TOOL_NAME, name)
                span.set_attribute('tool.input', f'args={args} kwargs={kwargs}')
                result = fn(*args, **kwargs)
                span.set_attribute('tool.output', str(result))
                return result
        return wrapper
    return decorator


def otel_retriever(name):
    """Wrap a function as an OTel span representing a retrieval step."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            query = args[0] if args else kwargs.get('query', '')
            with tracer.start_as_current_span(name) as span:
                span.set_attribute('retrieval.query', str(query))
                result = fn(*args, **kwargs)
                if isinstance(result, list):
                    span.set_attribute('retrieval.document_count', len(result))
                span.set_attribute('retrieval.output', str(result))
                return result
        return wrapper
    return decorator


def otel_span(name):
    """Generic OTel span wrapper for workflow/chain steps."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            with tracer.start_as_current_span(name) as span:
                span.set_attribute('span.input', f'args={args} kwargs={kwargs}')
                result = fn(*args, **kwargs)
                span.set_attribute('span.output', str(result))
                return result
        return wrapper
    return decorator


print('Decorators defined: @otel_tool, @otel_retriever, @otel_span')

Decorators defined: @otel_tool, @otel_retriever, @otel_span


## Step 6: Use the Decorators in a Workflow

With `@otel_retriever`, `@otel_tool`, and `@otel_span`, a multi-step pipeline becomes readable:

```python
@otel_retriever('search-knowledge-base')
def search_knowledge_base(query): ...

@otel_tool('format-context')
def format_context(docs): ...

@otel_span('knowledge-qa-pipeline')
def answer_question(query):
    docs = search_knowledge_base(query)                # retriever span
    context = format_context(docs)                      # tool span
    response = client.chat.completions.create(...)      # auto-instrumented LLM span
    return format_final_output(response)                # tool span
```

Each function carries its own span attributes, and the call graph becomes the trace hierarchy.

In [8]:
@otel_retriever('search-knowledge-base')
def search_knowledge_base(query):
    """Simulated vector search."""
    return [
        'TracerProvider is the central hub for OTel tracing.',
        'BatchSpanProcessor + OTLPSpanExporter is the standard production setup.',
        'OTel reads OTEL_EXPORTER_OTLP_ENDPOINT and HEADERS from env automatically.',
    ]


@otel_tool('format-context')
def format_context(docs):
    return '\n'.join(f'- {doc}' for doc in docs)


@otel_tool('format-final-output')
def format_final_output(answer):
    return answer.strip()


@otel_span('knowledge-qa-pipeline')
def answer_question(query):
    docs = search_knowledge_base(query)
    context = format_context(docs)

    llm_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer concisely based on the context provided.'},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'},
        ],
        max_tokens=100,
    )

    return format_final_output(llm_response.choices[0].message.content)


final = answer_question('What is a TracerProvider?')

# Flush both pipelines so the full trace + content events appear together.
tracer_provider.force_flush()
logger_provider.force_flush()

print(f'Answer: {final}')
print(f'\n→ Pipeline trace + content events flushed')
print(f'  Check: {log_stream_url}')

Answer: A TracerProvider is the central hub in OpenTelemetry (OTel) for managing and creating tracers, which are used to instrument applications for tracing purposes. It facilitates the configuration of tracing options and the processing of spans, typically using the BatchSpanProcessor along with exporters like OTLPSpanExporter for exporting trace data.

→ Pipeline trace + content events flushed
  Check: https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8/log-streams/7e8bce39-8429-4f4d-be58-42a7e894e931


## Step 7: Flush and Verify

A final flush before shutdown drains **both** pipelines:

- `tracer_provider.force_flush()` — exports buffered spans via `BatchSpanProcessor`
- `logger_provider.force_flush()` — drains pending log events through the `SpanAttrBridgeProcessor`

This is the OTel equivalent of `galileo_context.flush()`. In production code, this belongs in a shutdown hook (e.g. `atexit`).

In [ ]:
# Flush both signals before shutdown
tracer_provider.force_flush()
logger_provider.force_flush()

print('All spans + log events flushed to Galileo')
print(f'View traces and metrics at: {log_stream_url}')

## Step 8: Clean Up

Proper OTel shutdown is a three-step dance:

1. `tracer_provider.shutdown()` — stops the batch-export thread and flushes any remaining spans
2. `logger_provider.shutdown()` — same for the logs pipeline
3. `OpenAIInstrumentor().uninstrument()` — restores the original unwrapped `openai` library

We also delete the Galileo project so the notebook can be re-run cleanly. **Only run when done exploring** — deleted projects cannot be recovered.

In [ ]:
# OTel standard cleanup — shut down both signal pipelines
tracer_provider.shutdown()
logger_provider.shutdown()
OpenAIInstrumentor().uninstrument()

# Galileo platform cleanup
delete_project(name=PROJECT_NAME)
print(f'Deleted project: {PROJECT_NAME}')